In [1]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent / "utils"))

from transformers import AutoTokenizer
from unembeddings import load_lm_head_weight, find_closest_tokens
from animals_utils import get_animals

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
lm_head_weight = load_lm_head_weight(MODEL_NAME)

In [4]:
BOS_LENGTH = len(tokenizer("").input_ids)

def is_single_token(text):
    return len(tokenizer(text).input_ids[BOS_LENGTH:]) == 1

animals_raw = get_animals(MODEL_NAME, animal_set="synonyms")
# use space-prefixed singulars, matching how they appear in generation context
animal_texts = [f" {singular}" for singular, _ in animals_raw]

single_token_animals = [t for t in animal_texts if is_single_token(t)]
multi_token_animals  = [t for t in animal_texts if not is_single_token(t)]

print(f"Single-token: {single_token_animals}")
print(f"Multi-token:  {multi_token_animals}")

Single-token: [' elephant', ' dolphin', ' panda', ' lion', ' mosquito', ' rabbit', ' bunny', ' hare', ' snake', ' serpent', ' pig', ' hog', ' cougar', ' dove', ' pigeon', ' buffalo']
Multi-token:  [' kangaroo', ' penguin', ' giraffe', ' chimpanzee', ' koala', ' orangutan', ' cockroach', ' swine', ' puma', ' donkey', ' burro', ' ladybug', ' ladybird', ' bison']


In [5]:
results = {}
for animal in single_token_animals:
    results[animal] = find_closest_tokens(animal, tokenizer, lm_head_weight, top_k=5)

for animal, neighbors in results.items():
    print(f"\n{animal!r}")
    for token_text, sim, vec in neighbors:
        print(f"  {sim:.4f}  {token_text!r}  (vector shape: {vec.shape})")


' elephant'
  0.6665  ' Elephant'  (vector shape: (3584,))
  0.6420  ' elephants'  (vector shape: (3584,))
  0.4426  'Ele'  (vector shape: (3584,))
  0.4160  'ele'  (vector shape: (3584,))
  0.3950  ' ele'  (vector shape: (3584,))

' dolphin'
  0.5872  ' dolphins'  (vector shape: (3584,))
  0.5161  ' Dolphin'  (vector shape: (3584,))
  0.4018  ' Dolphins'  (vector shape: (3584,))
  0.2503  ' whale'  (vector shape: (3584,))
  0.2430  ' shark'  (vector shape: (3584,))

' panda'
  0.5725  ' Panda'  (vector shape: (3584,))
  0.3613  '熊猫'  (vector shape: (3584,))
  0.3303  ' pandas'  (vector shape: (3584,))
  0.2880  ' Pand'  (vector shape: (3584,))
  0.2674  ' pand'  (vector shape: (3584,))

' lion'
  0.6579  ' Lion'  (vector shape: (3584,))
  0.6479  ' lions'  (vector shape: (3584,))
  0.4859  'lion'  (vector shape: (3584,))
  0.4593  '狮子'  (vector shape: (3584,))
  0.4333  ' Lions'  (vector shape: (3584,))

' mosquito'
  0.6195  ' mosquitoes'  (vector shape: (3584,))
  0.5195  ' Mos'  (